# Genomics on the command line day 2: alignment files

## Setting up the workshop

## Alignment files
In the previous workshop we discussed a common sequence file formats, FASTA/Q, where each entry in the file represents a signle biological sequence (which could be a chromosome, a sequencing read, etc.). Such files are often a "first step" or the starting point in a bioinformatic genomic workflow. In this workshop, we will continue on in our workflow to discuss some common types of alignment files, which can be generated in a variety of ways and like sequence files can represent a variety of things: whole genome alignments to identify rearrangements, RNAseq reads vs reference transcriptome to calculate differential expression, genomic resequencing mapped to a genome to identify polymorphism, even certain sequencers will output reads in an alignment format. 

Commonly, alignment files have a "reference" (sometimes also called a "target") sequence, with "query" sequence(s) mapped against the reference. In this case, mapping coordinates (i.e. where the align to) are relative to the reference sequence. For example, in an RNAseq experiment the reads in the individual RNAseq samples would be the **query** sequences that are mapped against a **reference** transcriptome.

In terms of file format, alignment files come in many types. A variant of FASTA files, which we discussed in the previous workshop, can even be used for alignments. In FASTA alignments, each sequence is aligned against every other sequence, and all sequences are made the same length by adding gaps (either `N` or `-` characters, frequently). Thus the sequence in an alignment FASTA file represents the consensus between all the sequences in the alignment.

This type of alignment file is useful for certain basic tasks (e.g. aligning gene sequences from different species to make a phylogenetic tree), but other file types are more specialized to represent alignments and are more information dense. Let's look at...


## Intro to SAM/BAM format
SAM (Sequence Alignment/Map) format is one of the most common file formats produced by many different pieces of alignment software, both for long and short read sequence data. A number of different programs can output alignments in this format (e.g. [BWA](https://github.com/lh3/bwa), [STAR](https://github.com/alexdobin/STAR), [minimap2](https://github.com/lh3/minimap2)) and which you choose will vary based on your data type and experimental design, but the alignment file created will likely be interchangeable. In all cases, one sequence is designated as the reference, with queries aligned against it.

It is a tab delineated text file, with 11 mandatory fields, or columns, (listed below), with a 12th column containing optional "tags" with supplemental or aligner-specific information. SAM files are human readable, but can be quite large. An alternate format is the Binary Alignment/Map (BAM) file, which is binary compressed and not human readable, but is more compact and efficient to work with. Most pipelines will use BAM format over SAM, and for storing alignments long-term BAM is usually preferable as it uses up less storage space. Converting between BAM and SAM is easy, so there is no need to have both versions.

| **Column** | **Description**                        |
|------------|----------------------------------------|
| 1          | Read name                              |
| 2          | Bitwise flag                           |
| 3          | Reference name                         |
| 4          | Leftmost mapping position              |
| 5          | MAPQ quality score                     |
| 6          | CIGAR string                           |
| 7          | Name of 2nd read in pair               |
| 8          | Position of 2nd read in pair           |
| 9          | Length of mapping segment              |
| 10         | Sequence of segment                    |
| 11         | Phred33 quality score at each position |
| 12         | Optional tags                          |

In addition to these tab-separated fields, BAM/SAM files also have header lines at their starts, which are denoted by a `@` character and a two letter code. Header lines contain metadata about the alignment, such as whether alignments are sorted or grouped, reference sequence information, and definitions for any tags that are used in the 12th column. While not strictly required, many downstream analysis tools require BAM/SAM files to have a header, and will thrown an error if not present. A full description of BAM/SAM format can be found [here](https://samtools.github.io/hts-specs/SAMv1.pdf).

Many different types of bioinformatic analysis will use BAM/SAM alignments, such as:
- dssdg

Let's take a look at what these files look like.

> Use `less` to open SAM file (using `-S` to wrap lines). Do the same for BAM file.

In [ ]:
less -S ph.sam
less -S ph.bam

Just like any file, we can use our normal command line tools to visualize SAM files...however, it is a little clunky. Also, notice that when we try to open the BAM file, we get a scrambled mess; this is because as mentioned, BAM is a binary compressed format and is not human readable (which the shell will warn you of when trying to open the file). Thus, when working with BAM/SAM alignments, it is much easier to use a toolkit specilized for these types of files!

## SAMtools
[SAMtools](http://www.htslib.org/doc/samtools.html) is a suite of programs that are extremely useful for processing mapped reads and for downstream analysis. As stated above, BAM/SAM files from different programs are (mostly) interchangeable, so `samtools` will work with a file BAM/SAM file no matter what program produced it. Note that for this workshop we have already installed `samtools`, but to run it elsewhere you will need to install it yourself. It has a ton of functions (which you can check out on the [manual page](http://www.htslib.org/doc/samtools.html)), but we will go through several of the most common uses.

### samtools view
As the name suggests, this command lets you view the content of a SAM **or** BAM file. Let's take a look at a file by opening it with `view` and piping it to `head` to display just the first five lines.

In [ ]:
samtools view ph.bam | head -n 5

Note that even tho we are opening a BAM file, `view` automatically converts it into readable SAM format! This is how we convert BAM --> SAM. To convert SAM --> BAM, we can use the `-b` argument to instead output in BAM format, along with the `-o` argument to output to a file instead of STDOUT.

```
samtools view -b -o converted.bam ph.sam
```

However, by default something important is missing when we save output to a file this way...can you figure out what it is?

<details><summary>Solution</summary>

The output file lacks a header! As mentioned, headers are vital for many downstream tools. 

> **Exercise** 
> Check the samtools documentation and find the correct option to add a header
  
</details>

In [ ]:
# command here

In [ ]:
#@title Solution {display-mode: "form"}
samtools view -h -b -o converted.bam ph.sam

#### Building intuition: what's wrong with my BAM file?
Just like we did in the previous workshop, we want to try to work on cultivating that bioinformatic "sixth sense" that helps us identify when something doesn't look right and where/how to troubleshoot. Doing this with BAM/SAM files is tricky, as the format is more complex and there are many areas that problems can arise. Broadly, I would categorize errors in BAM files as falling into two categories:

- **Format errors**: something is wrong your `samtools` syntax or with the file itself, e.g.:
  - Truncated or malformatted files
  - Missing header or info in the header
  - Confusing SAM and BAM
- **Alignment errors**: something is wrong with your data, e.g.:
  - Reference or queries are incorrect
  - A mistake when generating the alignment (e.g. used wrong settings)
  - Your data is bad

Periodically throughout this workshop, we'll walk through some hypothetical situations you might find yourselves in when working with these kinds of files and how to diagnose and the fix the problems!


In this scenario, imagine you have a SAM file of RNAseq data aligned to a reference transcriptome that you are trying to use to calculate differential expression between different tissues. You convert the SAM file to BAM format and feed the BAM file into a software pipeline that will calculate expression... but the program gives an error `"the specified BAM file is empty!`

```
samtools view -H -o file.bam file.sam
```

**Discussion**: what is a logical place to start troubleshooting the issue?

#### SAM flags and filtering
The second column in a BAM/SAM file is the *bitwise flag*. The flag value is an integer, which is the sum of a series of decimal values that give information about how a read is mapped.

| **Integer** | **Description**                |
|-------------|--------------------------------|
| 1           | read is paired                 |
| 2           | read mapped in proper pair     |
| 4           | read unmapped                  |
| 8           | mate is unmapped               |
| 16          | read on reverse strand         |
| 32          | mate on reverse strand         |
| 64          | first read in pair             |
| 128         | second read in pair            |
| 256         | not primary alignment          |
| 512         | alignment fails quality checks |
| 1024        | PCR or optical duplicate       |
| 2048        | supplementary alignment        |

So e.g., for a paired-end mapping data set, a flag = **99** (1+2+32+64) means the read is mapped along with its mate (1 and 2) and in the proper orientation (32 and 64). Don't worry about memorizing these, there are plenty of tools online that decode these flags for you, such as right [here](https://broadinstitute.github.io/picard/explain-flags.html).

While you don't need to know all the SAM flags, if there is one flag that is useful to have memorized it is `4`, which means the read is **unmapped**. Unmapped reads are most often filtered out, as many programs used in downstream analysis of SAM/BAM files only want mapped reads (and also to save space on disk!). You can filter reads containing a given flag using the `-f` (only take reads that match given flags) and `-F` (only take reads that do **NOT** match given flag) options in `samtools view`.

Note that filtering on a particular flag works not just for that literal flag, but for *any integer sum that contains that flag*. For example, `-F 4` will filter out any alignment with the `4` flag, but also e.g.: 

- `77` (1+4+8+64) = "read paired + read unmapped + mate unmapped + first in pair" 
- `141` (1+4+8+128) = "read paired + read unmapped + mate unmapped + second in pair"
- `101` (1+4+32+64) = "read paired + read unmapped + mate on reverse strand + first in pair"

> **Exercise**:
> Write a command that counts the number of unmapped reads in the file `data/file.sam`.

In [ ]:
## Command here

In [ ]:
#@title Solution {display-mode: "form"}
samtools view -f 4 data/file.sam | wc -l

> **Exercise**:
> In the code block below, count how many reads in the SAM file are mapped in their proper pairs vs not proper pairs?

In [ ]:
# command here

In [ ]:
#@title Solution {display-mode: "form"}

In [ ]:
#@title Solution {display-mode: "form"}
samtools view -f 2 data/file.sam | wc -l
samtools view -F 2 data/file.sam | wc -l

### Developing intuition: what's wrong with my BAM file?
Imagine you are studying nucelotide diversity within a population. A collaborator has generated a BAM alignment file by mapping whole genome sequencing data from 100 individuals in the population against a reference genome, which you want to use for analysis. You download the BAM file from your collaborator's lab server to your local machine and take a look at it:

```
ls -lh large_alignment.bam
```

**Discussion**: what do you notice that is suspicious?

<details><summary>Solution</summary>

Given that the BAM is supposed to be an alignment of a hundred samples, we would expect the file to be quite large, probably 100s of GB in size, and the file the we downloaded is *way too small*. When downloading files from a server (especially large files), it is not uncommon for a download to get interrupted and the file to be truncated. You will probably need to re-download the file.

To check your file, BAM/SAM files contain a 28-byte `E`nd `O`f `F`ile (`EOF`) marker at the end of the file to mark that the file is complete. `samtools` has a built-in tool that will verify the `EOF` is present:

```
samtools quickcheck file.bam
```

Also, when downloading a file from a server, frequently the file uploader will include a `checksum`. A `checksum` is essentially a unique "fingerprint" that is generated from a file using an algorithm that looks like a string of integers. If you know what algorithm was used to generate the checksum (e.g. `md5`, one of the most common algorithms), you can generate a fingerprint of your copy of the file and compare it to the uploader's to verify your file's integrity...if they do not match, it means your file was altered.
  
</details>

### Sorting and indexing a BAM file
BAM files can get extremely large, over hundreds of GB in some cases, and by default are unsorted, which makes any task where we need to search for specific regions too computationally intensive. To overcome this, we can use two other functions of `samtools`, `sort` and `index`. This will create a BAM index `.bai` file, which allows quick lookup even for very large BAM files.

In [ ]:
# -o: This option tells samtools sort to print the output to the provided file rather than to the screen
samtools sort -o file.sorted.bam file.bam

#samtools index does not have an -o option, and will automatically create an index file with the same name as the input BAM file with a .bai extension
samtools index file.sorted.bam

This code will create a new `.sorted.bam` file that we create the index for, as `samtools index` requires a coordinate-sorted file. Any downstream program that refers to specific regions of interest in a BAM file, such as visualization tools like IGV (discussed later) or other `samtools` functions will require this index. For example, we can specify *specific region(s)* when using `samtools view` to only print alignments that overlap the specified region. Regions are listed at the end of the `view` command with the format `reference name:start position-end position` (if start and end are not specified, it reports all alignments to the reference sequence): 

In [ ]:
#Outputs all alignments that are mapped to chromosome 1 and saves them to a new BAM file
samtools view -o ph.chr1.bam ph.bam chr1

#Outputs all alignments that are mapped from bases 1000 to 2000 (inclusive) on chromosome 1 and saves them to a new BAM file
samtools view -o ph.chr1:1000-2000.bam ph.bam chr1:1000-2000

> PRACTICE: Putting it all together!

Let's take everything that we have learned and organize it into what a typical workflow might look like. Assume that we have already aligned the data and the aligner we used outputs the alignment file in SAM format. We want to go from our initial SAM file and end up with a *sorted, indexed BAM file with only the mapped reads retained*. Try inputting the commands yourself, then we will walk through it together.

> **Exercise**:
> 1. Convert the `data/file.sam` **SAM** file to **BAM** format while retaining the header and removing unmapped reads, then sort the file. Call the new file `file.mapped.sorted.bam`.
> Bonus: `samtools` commands can also make use of **pipes** (`|`) to avoid writing intermediate files!
> 2. Index the newly created sorted **BAM** file.

In [ ]:
## Command here

In [ ]:
#@title Solution {display-mode: "form"}

In [ ]:
#@title Solution {display-mode: "form"}

## Convert to BAM with header and without unmapped reads and then sort
samtools view -h -b -o file.mapped.bam -F 4 data/file.sam
samtools sort -o file.mapped.sorted.bam file.mapped.bam

#Or, in a single line:
samtools view -h -b -F 4 data/file.sam | samtools sort -o file.mapped.sorted.bam 

## Index the new BAM file
samtools index file.mapped.sorted.bam

### More useful `samtools` utilities
We have our nice sorted and indexed BAM file, now what are some other useful pieces of information we can pull out of it? 

#### `samtools stats`
As the name suggests, the `stats` function calculates some basic summary statistics about a BAM file, such as number of sequences aligned, number of reads mapped in proper pairs (if using paired-end data), average alignment error rate, average alignment quality, plus lots more which are all described in the [manual page](https://www.htslib.org/doc/samtools-stats.html). 

```
samtools stats ph.bam > ph.bam.stats
```

> **Exercise**
> We are only interested in the "summary numbers" section of the stats output file. Thinking back to last week's workshop, write a command that pulls the summary numbers out of `samtools stats` output (hint: check the manual page or look thru the file to find the relevant pattern)

In [ ]:
## Command here

In [ ]:
#@title Solution {display-mode: "form"}
samtools stats ph.bam | grep '^SN' > ph.bam.stats.txt

#### Converting to sequence files
We know that BAM/SAM files contain sequence names, nucleotide sequence and corresponding quality scores...in other words, everything we need to make a FASTA/FASTQ file! Accordingly, we can use the built-in function `samtools fastq` and `samtools fasta`:

```
samtools fastq -o output.fastq file.bam
```

This would output to a single outfile, such as when we have unpaired reads. For paired end reads, we would instead want:

```
samtools fastq -1 reads1.fastq -2 reads2.fastq file.bam
```

> **Exercise**:
> Like any other `samtools` functions, we can use `fastq/a` in combination with other tools. Write a command that output only the mapped reads in a BAM file to a FASTQ file (BONUS: only output primary, non-supplementary alignments). The reads are unpaired.

In [ ]:
## Command here

In [ ]:
#@title Solution {display-mode: "form"}
samtools view -F 4 file.bam | samtools fastq -o mapped_reads.fastq

#Bonus: 4 + 256 + 2048 = 2308 (unmapped, not primary alignment, and supplementary alignment flags)
samtools view -F 2308 file.bam | samtools fastq -o mapped_reads.fastq

### Calculating read coverage
In many kinds of bioinformatic analysis, we will be interested in the **read coverage** and **read depth** in a particular region. "Coverage" and "depth" are often used interchangably but are technically are different concepts. "Coverage" is defined as the percentage of positions that have *at least one base aligned to it* (think of it as how much sequence is covered by mapped reads), while "depth" can be thought of as the redundancy of coverage (i.e. how many bases are aligned to a particular sequence). Aligning vs a reference and calculating coverage and/or depth for an alignment is the starting point for mainy kinds of analysis, such as:

- Identifying misassembled regions in a genome assembly
- Calculating differential expression of transcripts
- Calling single nucleotide polymorphisms or structural variants within a population 

To calculate this, we can use (appropriately enough) `samtools coverage` and `samtools depth`! As we can see from the [manual page](https://www.htslib.org/doc/samtools-coverage.html), the `coverage` command calculates a summary of both coverage and depth within a specified region, or over the entirety of each reference sequence if no regions are provided. E.g.:

In [ ]:
samtools coverage ph.bam

In [ ]:
samtools coverage -r chr1:1000-2000 ph.bam

Note that unlike with `samtools view` where we specify a region at the end of the command, `coverage` requires it as an argument with `-r`...this is a good reminder that unfortunately command syntax won't always be consistent (even within the same software package!), and to always check the manual!

While a summary of coverage/depth is useful, often we will want to know to know depth at each *individual position* in the reference, as that will to identify specific loci where depth is abnormally high or low. For this we use `samtools depth`, which will return a 3 column list of reference sequence, numeric position (i.e. base 1, 2, 3, etc.) and the depth at that base:

In [ ]:
#The -a argument tells samtools to output depth at all positions, including positions with zero depth
#-o outputs the list to a file instead of the terminal
samtools depth -o ph.bam.depth.txt -a ph.bam

head ph.bam.depth.txt

### Developing intuition: what's wrong with my BAM file?
You have a reference genome assembly and whole genome sequencing from a different individual of the same species. You are interested in identifying structurally complex regions on a specific chromosome using read coverage, so you run a pipeline that aligns the sequencing reads against your reference to create a BAM file, pulls out out alignments specific to the chromosome of interest, calculates coverage depth and generates a plot of coverage depth across the chromosome for visual inspection. The pipeline runs successfully, and the coverage plot looks like this:

**Plot showing zero coverage across whole chromosome**

**Discussion**: what is the problem here? What are some possible things that could be going wrong? Where should we start in troubleshooting?

<details><summary>Solution</summary>

We are seeing zero coverage across the chromosome, which definitely is not right!

The actual problem could be from several different sources:
- We aligned the wrong data/used the wrong reference
- The BAM file is corrupted/empty
- The visualization script is doing something wrong
- Something is wrong with the sequencing data itself

Here are the steps I would take when troubleshooting:
- Inspect the BAM file using `samtools view`, make sure there is stuff in it
- Check the commands run:
  - Are the data and reference correct?
  - Was there some filtering done at any step?
  - Does my plotting script work properly?
After this, I'd start worrying about the data itself...
  
</details>

The commands that were run as part of the pipeline are as follows:

```
#Alignment step
minimap2 reference.fasta reads.fastq > reads_vs_reference.sam

#Convert to BAM and index
samtools view -b -h -f 4 -o reads_vs_reference.mapped.bam reads_vs_reference.sam

samtools sort -o reads_vs_reference.mapped.sorted.bam reads_vs_reference.mapped.bam
samtools index reads_vs_reference.mapped.sorted.bam

#Subset alignment and calculate depth
samtools view -h -b -o chr1.bam reads_vs_reference.mapped.sorted.bam
samtools depth chr1.bam > chr1.depth.txt

#Make plot
Rscript make_coverage_plot.R chr1.depth.txt > chr1.depth.png

```

<details><summary>Solution</summary>

Looking at our pipeline, let's say that the alignment step is correct and the plotting script should work correctly.

Checking our BAM file, we can see that there are entries in it...

```
samtools view reads_vs_reference.mapped.sorted.bam | less
```

However! Looking closer, we can see that the reads in the BAM file *are all unmapped*. And when we look at the `view` command in our pipeline, we can see that the source of the error is due to a mistake in filtering: we were trying to remove unmapped reads by filtering on the `4` flag, but we used the *wrong argument*.

We wanted `-F 4` (which will exclude alignments with the `4` flag) but instead we used `-f 4` (which only includes alignments with the `4` flag)!

This is a classic mixup when working with BAM files, and shows how a subtle syntax error can cause a problem that is only noticed at a later point.
  
</details>

In the above example, we are using the external program `R` to visualize coverage over our reference from the file created by `samtools depth`, which we could also accomplish using a `python` script or some other language like `matlab`. While the detail can vary based on what kind of analysis you are doing, this is a common approach when plotting read depth or coverage, as making plots at the command line is very clunky. However, let's explore one other method we can use to visualize coverage.

### Visualizing alignments with IGV
Although it isn't technically a command line program and is not part of `samtools` or even limited to just alignment files, I wanted to introduce the `I`ntegrative `G`enome `V`iewer (`IGV`) program, as it is probably the most widely used graphical user interface to visualize BAM files. Also, while writing your own scripts for vizualization in `R` or `python` is useful, sometimes have an interactive interface to explore alignments and "eyeball" your what they actually look like is essential. `IGV` is its own standalone program (which you can download [here](https://igv.org/doc/desktop/), though it also requires `java` to be installed) and has way too much functionality to go over fully, but we will go over the basics of visualizing a BAM alignment.

Once we have installed and opened `IGV`, we will need three files: the reference genome in FASTA format, the BAM alignment we want to vizualize and the corresponding index file (which should have the same named prefix as the BAM plus the `.bai` extension and be in the same directory).

- First click the `Genomes` tab --> `Load Genome from File` --> navigate to and select the genome FASTA file
- Click `File` --> `Load from File` --> navigate to and select the BAM file

We should see a track with representations of reads mapped to the reference and a histogram showing depth of coverage. We can select different reference chromosomes or specific regions using the menu bars at the top center, use the mouse to pan along the sequence, zoom in and out, etc. We can also look at individual mapped reads, which have bases colored whether they match reference (by default, grey = same as reference) as well as any gaps or indels. 

Again, this is just a super quick overview of what `IGV` can do to get you aware that it is an option! 